# editor for `avg_vol` fcn

In [1]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
from nilearn import masking # for masking within-brain voxels

import nitools as nt

import os

In [2]:
# directories
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [3]:
tissue = 'wm' # default
tissue_dict = {
    'gm': 'c1',
    'wm': 'c2',
    'csf': 'c3'
}

In [4]:
from pathlib import Path

In [18]:
subj_id = 'CU_2538'
week = 'W0'

results_path = Path(anat_dir)/subj_id/week/'iso_norm_mniSymm/' # folder specifies space

t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'
mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'


In [19]:
reference_img = f'{anat_dir}/{subj_id}/{week}/c2{subj_id}_W0_T1.nii'

In [ ]:

"""
Inputs:
anat dir: participants file OR [(subj, week) and call it inside a loop].
Reference img (inside the Jupyter notebook loop for reading off the info file)
#week_path (path for each week's image), results_path (store results)
results path: directory to store results
image suffix: suffix with which to save the slope and intercept images
    suggested: <image_type>_<space> where 'image_type' is "anat", "wm", "gm", etc; 'space' is native or template (<template_name>)

Everything is done in the reference image. So this function will (...) (resample voxels in other weeks so that they are aligned with the reference, and perform multiple linear regression)

Returns B_hat coefficient matrix (for more flexibility in other possible operations)

"""

# file prefix encoding (based on SPM segmentation notation)
tissue_dict = {
    'gm': 'c1',
    'wm': 'c2',
    'csf': 'c2'
}

img0 = nib.load(reference_img)

# later fix: option to reduce to only wtihin-brain voxels

# transform into world coordinates
i, j, k = np.indices(img0.shape) # matrix indices for premult by affine
x,y,z = nt.affine_transform(i, j, k, img0.affine)

# all possible weeks.
weeks = np.array([0,4,12,24,52]) # read from file, use file reading function maybe


# THIS PART SHOULD BE DONE IN TUTORIAL, NOT IN FUNCTION. or with a helper function.


#____________________________________
# find the number of measurement weeks that exist
p_weeks = []
for week in weeks:
    #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
    #week_path = week_path
    #week_path = f'{anat_dir}/{subj_id}/W{week}/c2{subj_id}_W{week}_T1.nii' # fix

    if not tissue==None:
        week_path = f'{anat_dir}/{subj_id}/W{week}/{tissue_dict[tissue]}{subj_id}_W{week}_T1.nii'
    else:
        week_path = f'{anat_dir}/{subj_id}/W{week}/{subj_id}_W{week}_T1.nii'


    # skip over missed measurement weeks.
    if not os.path.exists(week_path):
        continue

    p_weeks.append(week)

#__________________________


Y = np.zeros((len(p_weeks), np.prod(img0.shape))) # initialize Y (shape = (k by p)) array, where k = number of weeks available

for week in weeks: # resample ALL weeks, including the reference week
    #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
    #week_path = week_path
    
    if not tissue==None:
        week_path = f'{anat_dir}/{subj_id}/W{week}/{tissue_dict[tissue]}{subj_id}_W{week}_T1.nii'
        #print(f'using {tissue}')
    else:
        week_path = f'{anat_dir}/{subj_id}/W{week}/{subj_id}_W{week}_T1.nii'


    # skip over missed measurement weeks.
    if not os.path.exists(week_path):
        continue

    week_img = nib.load(week_path)
    print(f'on week {week} for {subj_id}')

    week_dict = {
        '0': 0,
        '4': 1,
        '12': 2,
        '24': 3,
        '52': 4
    }


    # resample each week's image so that voxels are exactly on top of reference week voxels; add to response matrix as row vector
    Y[week_dict[str(week)]:,] = nt.sample_image(week_img, # response matrix
                            xm=x, ym = y, zm = z, # world coordinates
                            interpolation = 1 # using trilinear resampling
                            ).flatten() # need to put each week as a row
    
    # now we have Y as a k by p matrix, where k is the number of weeks.  

# design matrix
num_weeks = len(p_weeks)
X = [np.ones(shape = (num_weeks)), p_weeks]
X = np.array(X)
X = X.T

# estimator (coefficients matrix)
B_hat = np.linalg.pinv(X) @ Y
# where B_hat = [B_0 B_1].T

#_________________
# save image with voxel coordinates
slope = np.zeros(img0.shape) # tensor with shape of reference img
intercept = np.zeros(img0.shape)
# need i, j, k as vectors (they're tensors right now)

iv = i.flatten()
jv = j.flatten()
kv = k.flatten()

slope[iv, jv, kv] = B_hat[1,:] # write the slope into the vectors i, j, k for coordinates
intercept[iv, jv, kv] = B_hat[0,:]
#________________

# save as Nifti
intercept_img = nib.Nifti1Image(intercept, img0.affine)
slope_img = nib.Nifti1Image(slope, img0.affine)

nib.save(intercept_img, f'{results_path}/{subj_id}_T1_intercept_{image_suffix}.nii.gz') # specify file name
nib.save(slope_img, f'{results_path}/{subj_id}_T1_slope_{image_suffix}.nii.gz')

"""
results = np.zeros(img0.shape)
results(i,j,k)=B(:,1) # Slopw
nifti = np.Nifti1image(results,img0.affine)
niftt.to_filename()

"""

# intercept and slope reshaped into Nifti-compatible array
#intercept = B_hat[0,:].reshape(img0.shape)
#slope = B_hat[1,:].reshape(img0.shape)

"""
So this image is in world coordinates now, and saved with the affine of the original image.
Should it be converted back to voxel coordinates (bc otherwise, the affine is kinda meaningless)?

# intercept and slope are already world-coordinates array, so multiply by inverse affine

"""

"""
# save as Nifti
intercept_img = nib.Nifti1Image(intercept, img0.affine)
slope_img = nib.Nifti1Image(slope, img0.affine)

# fix: name of file should be insertable, too (e.g. which_type = 'native' or smth in fcn input)
nib.save(intercept_img, f'{results_path}/{subj_id}_T1_intercept_{image_suffix}.nii.gz') # specify file name
nib.save(slope_img, f'{results_path}/{subj_id}_T1_slope_{image_suffix}.nii.gz')
"""


on week 0 for CU_2538
on week 4 for CU_2538


AttributeError: module 'numpy' has no attribute 'Nifti1Image'

In [36]:
nifti_slope = nib.Nifti1Image(results, img0.affine)

In [38]:
nib.save(nifti_slope, 'voxel_slope_CU_2538_trial.nii.gz')

In [34]:
results.max()

np.float64(0.2500000147847458)

In [31]:
results[iv]

MemoryError: Unable to allocate 5.31 TiB for an array with shape (11141120, 256, 256) and data type float64

In [25]:
iv

array([  0,   0,   0, ..., 169, 169, 169], shape=(11141120,))

In [17]:
results

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0.

In [5]:
results

NameError: name 'results' is not defined